# Interface statistics: elevation and slope

Windowed statistics of the air--water interface $\eta(x,z,t)$ from a wind-wave
simulation, together with the running PDFs of the normalised elevation
$\xi = (\eta - \langle\eta\rangle)/\sigma_\eta$.

This notebook is the interactive counterpart of `generate_eta_plots.py`, which produces the
same figures headlessly (it is what the run script calls on the cluster). Everything here
reads the small text tables in `statistics/`; none of the large field dumps are needed.

**Inputs**

| file | contents |
| --- | --- |
| `statistics/eta_stats_window.out` | one row per averaging window: moments of $\eta$, $\partial_x\eta$, $\partial_z\eta$ |
| `statistics/eta_pdf_window_*.out` | binned PDF of $\xi$ and $\eta$ for each window |
| `case_params.txt` (or `out.log`) | case parameters; supplies the peak period $T_p \equiv T_0$ used to non-dimensionalise time |

## 1. Setup

In [ ]:
%matplotlib inline

import glob
import io
import os
import re

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.rcParams.update({
    "figure.figsize": (11, 6),
    "figure.dpi": 110,
    "savefig.dpi": 220,
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "axes.linewidth": 1.0,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "grid.linewidth": 0.8,
    "lines.linewidth": 2.0,
    "lines.markersize": 5,
})


def style_axes(ax):
    """Major + faint minor grid, matching the headless script."""
    ax.grid(True, which="major")
    ax.minorticks_on()
    ax.grid(True, which="minor", alpha=0.10, linestyle=":", linewidth=0.6)

### Where the data lives

`CASE_DIR` is the only thing to change. It works unmodified in two places:

* inside the simulation directory on the cluster, where `statistics/` sits next to `out.log`;
* inside this repository, where a case has been copied to `data/<case-name>/`.

In [ ]:
# Set explicitly to override the search below, e.g.
#   CASE_DIR = "/scratch/gpfs/DEIKE/cm6797/multiphase_cases/re720_bo200_..."
CASE_DIR = None

# Candidate roots, in order of preference.
_candidates = []
if CASE_DIR:
    _candidates.append(CASE_DIR)
_candidates += [".", ".."]
_candidates += sorted(glob.glob(os.path.join("data", "*")))
_candidates += sorted(glob.glob(os.path.join("..", "data", "*")))

CASE_DIR = next(
    (os.path.abspath(c) for c in _candidates
     if os.path.isfile(os.path.join(c, "statistics", "eta_stats_window.out"))),
    None,
)
if CASE_DIR is None:
    raise FileNotFoundError(
        "No statistics/eta_stats_window.out found. Set CASE_DIR to the simulation "
        f"directory by hand. Looked in: {_candidates}"
    )

STATS_DIR = os.path.join(CASE_DIR, "statistics")
PLOTS_DIR = os.path.join(STATS_DIR, "plots")

# Write the figures to statistics/plots/ as well as showing them inline.
SAVE_FIGS = False

print(f"case  : {CASE_DIR}")
print(f"stats : {STATS_DIR}")

In [ ]:
def load_table(path):
    """Read a whitespace-separated table, tolerating literal "\\n" written by older runs."""
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            content = f.read()
        if "\\n" in content:
            content = content.replace("\\n", "\n")
        data = np.loadtxt(io.StringIO(content), comments="#")
    except (OSError, ValueError):
        return np.empty((0, 0))
    if data.ndim == 1:
        data = data.reshape(1, -1)
    return data


def infer_tp(case_dir):
    """Recover the peak period T_p = T0 from the run's parameter dump."""
    for name in ("case_params.txt", "out.log", "log_simulation.out"):
        path = os.path.join(case_dir, name)
        if not os.path.exists(path):
            continue
        try:
            with open(path, "r", encoding="utf-8", errors="replace") as f:
                txt = f.read()
        except OSError:
            continue
        m = re.search(r"T0\s*=\s*([0-9eE+\-.]+)", txt)
        if m:
            try:
                tp = float(m.group(1))
            except ValueError:
                continue
            if tp > 0.0:
                return tp, f"{name}"
    return 1.0, "fallback (time left dimensional)"


def savefig(fig, name):
    """Persist a figure to statistics/plots/ when SAVE_FIGS is on."""
    if not SAVE_FIGS:
        return
    os.makedirs(PLOTS_DIR, exist_ok=True)
    out = os.path.join(PLOTS_DIR, name)
    fig.savefig(out, dpi=180)
    print(f"wrote {out}")


TP, TP_SOURCE = infer_tp(CASE_DIR)
print(f"Tp = {TP:.10e}   (from {TP_SOURCE})")

## 2. Windowed moments

`eta_stats_window.out` is an *append-only trace of the running window accumulator*, not a
table of finished windows: every dump appends the moments accumulated so far in the window
currently being filled. `is_partial = 1` marks such an in-progress row and `is_partial = 0`
the closing row of a window, so a run with ten windows produces a few hundred rows of which
ten are final. Reading it as one-row-per-window makes early, poorly converged partial sums
look like real time variability — the toggle in the loading cell below lets you switch
between the two views.

Older runs wrote 13 columns (elevation only); runs with the slope diagnostics enabled
write 21. The slope moments are normalised over the interface area with
$|n_y| > 0.10$, i.e. near-vertical cells are excluded from the slope statistics.

| col | quantity | | col | quantity |
| --- | --- | --- | --- | --- |
| 0 | `t_end` | | 11 | $\kappa(\xi)$ kurtosis |
| 1 | `i` (step) | | 12 | window weight |
| 2 | `t_rel` | | 13--16 | $\langle s_x\rangle,\ \sigma_{s_x},\ \gamma_1(s_x),\ \kappa(s_x)$ |
| 3 | window id | | 17--20 | $\langle s_z\rangle,\ \sigma_{s_z},\ \gamma_1(s_z),\ \kappa(s_z)$ |
| 4 | `is_partial` | | | |
| 5 | `nsnap` | | | with $s_x=\partial\eta/\partial x$, $s_z=\partial\eta/\partial z$ |
| 6--7 | `t_start`, `t_last` | | | |
| 8--9 | $\langle\eta\rangle$, $\sigma_\eta$ | | | |
| 10 | $\gamma_1(\xi)$ skewness | | | |

Restarts also make the file replay rows it had already written, so exact duplicates are
dropped on load.

In [ ]:
# False -> plot the full running trace, with closed windows marked.
# True  -> keep only the closing row of each window (one point per completed window).
COMPLETE_ONLY = False

stats_file = os.path.join(STATS_DIR, "eta_stats_window.out")
raw = load_table(stats_file)
if raw.size == 0:
    raise RuntimeError(f"No numeric rows in {stats_file}")

ncol = raw.shape[1]
HAS_SLOPES = ncol >= 21

# Restarts replay rows that were already written: drop exact duplicates, keep file order.
_, keep = np.unique(raw, axis=0, return_index=True)
W = raw[np.sort(keep)]
ndup = raw.shape[0] - W.shape[0]

closed_all = W[:, 4].astype(int) == 0
if COMPLETE_ONLY:
    W = W[closed_all]

print(f"{raw.shape[0]} rows read, {ndup} duplicate(s) dropped -> {W.shape[0]} kept")
print(f"{ncol} columns (slope moments: {'yes' if HAS_SLOPES else 'no'}); "
      f"{closed_all.sum()} closed window(s) of {int(raw[:, 3].max())} started")

t_rel      = W[:, 2]
tau        = (t_rel - t_rel[0]) / TP          # elapsed time in peak periods
win_id     = W[:, 3].astype(int)
is_partial = W[:, 4].astype(int)
nsnap      = W[:, 5].astype(int)
is_closed  = is_partial == 0

eta_mean  = W[:, 8]
eta_sigma = W[:, 9]
xi_skew   = W[:, 10]
xi_kurt   = W[:, 11]

if HAS_SLOPES:
    sx_mean, sx_sigma, sx_skew, sx_kurt = W[:, 13], W[:, 14], W[:, 15], W[:, 16]
    sz_mean, sz_sigma, sz_skew, sz_kurt = W[:, 17], W[:, 18], W[:, 19], W[:, 20]

print(f"tau spans 0 -> {tau[-1]:.2f} Tp")

In [ ]:
# Closing row of each window: the converged numbers worth quoting.
rows = np.flatnonzero(is_closed)
if rows.size == 0:
    print("No window has closed yet - showing the last 10 partial rows instead.")
    rows = np.arange(max(len(tau) - 10, 0), len(tau))

hdr = (f"{'win':>5} {'tau':>8} {'nsnap':>7} {'<eta>':>11} {'sigma_eta':>11} "
       f"{'skew':>8} {'kurt':>8}")
if HAS_SLOPES:
    hdr += f" {'sig_sx':>9} {'sig_sz':>9} {'kurt_sx':>9}"
print(hdr)
print("-" * len(hdr))
for j in rows:
    line = (f"{win_id[j]:5d} {tau[j]:8.2f} {nsnap[j]:7d} {eta_mean[j]:11.4e} "
            f"{eta_sigma[j]:11.4e} {xi_skew[j]:8.3f} {xi_kurt[j]:8.3f}")
    if HAS_SLOPES:
        line += f" {sx_sigma[j]:9.4f} {sz_sigma[j]:9.4f} {sx_kurt[j]:9.3f}"
    print(line)

In [ ]:
def mark_windows(ax, y=None):
    """Mark where each averaging window closed; those rows are the converged ones."""
    for j in np.flatnonzero(is_closed):
        ax.axvline(tau[j], color="k", alpha=0.18, lw=0.9, ls="--")
        if y is not None:
            ax.plot(tau[j], y[j], marker="o", mfc="none", mec="k", ms=9, mew=1.2, alpha=0.8)


c_mean, c_sig, c_skew, c_kurt = "#1f4e79", "#d95f02", "#1b9e77", "#6a3d9a"

fig, axs = plt.subplots(3, 1, figsize=(11, 10), sharex=True, constrained_layout=True)

axs[0].plot(tau, eta_mean, "-o", color=c_mean, label=r"$\langle \eta \rangle_{\rm win}$")
axs[0].set_ylabel(r"$\langle \eta \rangle_{\rm win}$")
axs[0].legend(loc="best", frameon=False)
style_axes(axs[0])

axs[1].plot(tau, eta_sigma, "-o", color=c_sig, label=r"$\sigma_{\eta,\rm win}$")
axs[1].set_ylabel(r"$\sigma_{\eta,\rm win}$")
axs[1].legend(loc="best", frameon=False)
style_axes(axs[1])

axs[2].plot(tau, xi_skew, "-o", color=c_skew, label=r"$\xi_{\rm skew}$")
axs[2].plot(tau, xi_kurt, "-o", color=c_kurt, label=r"$\xi_{\rm kurt}$")
axs[2].axhline(0.0, color="0.3", lw=0.9, ls="--", alpha=0.7)   # Gaussian skewness
axs[2].axhline(3.0, color="0.45", lw=0.9, ls=":", alpha=0.7)   # Gaussian kurtosis
mark_windows(axs[2], xi_kurt)
axs[2].set_xlabel(r"$\tau = (t_{\mathrm{rel}} - t_{\mathrm{rel},0})/T_p$")
axs[2].set_ylabel(r"shape moments")
axs[2].legend(loc="best", frameon=False, ncol=2)
axs[2].xaxis.set_major_locator(MaxNLocator(nbins=8))
style_axes(axs[2])

axs[2].set_title(r"dashed lines / open circles: window closures", fontsize=11, color="0.35")
fig.suptitle(r"Windowed interface statistics", fontsize=20)
savefig(fig, "eta_stats_windows.png")
plt.show()

## 3. Surface-slope moments

The along-wind slope $s_x = \partial\eta/\partial x$ and the crosswind slope
$s_z = \partial\eta/\partial z$. Symmetry of the crosswind direction means
$\langle s_z\rangle$ and $\gamma_1(s_z)$ should sit at zero; any drift there is a
diagnostic of insufficient averaging rather than physics.

The cells below no-op if the run predates the slope columns.

In [ ]:
if not HAS_SLOPES:
    print(f"Slope columns not available (need 21 columns, got {ncol}) - skipping section 3.")
else:
    fig, axs = plt.subplots(2, 1, figsize=(11, 8), sharex=True, constrained_layout=True)

    axs[0].plot(tau, sx_mean, "-o", color="#1f4e79", label=r"$\langle s_x \rangle$")
    axs[0].plot(tau, sz_mean, "-s", color="#d95f02", label=r"$\langle s_z \rangle$")
    axs[0].axhline(0.0, color="0.4", lw=0.9, ls="--", alpha=0.7)
    axs[0].set_ylabel(r"mean slope")
    axs[0].legend(loc="best", frameon=False, ncol=2)
    style_axes(axs[0])

    axs[1].plot(tau, sx_sigma, "-o", color="#1f4e79", label=r"$\sigma_{s_x}$")
    axs[1].plot(tau, sz_sigma, "-s", color="#d95f02", label=r"$\sigma_{s_z}$")
    axs[1].set_xlabel(r"$\tau = (t_{\mathrm{rel}} - t_{\mathrm{rel},0})/T_p$")
    axs[1].set_ylabel(r"slope std")
    axs[1].legend(loc="best", frameon=False, ncol=2)
    axs[1].xaxis.set_major_locator(MaxNLocator(nbins=8))
    style_axes(axs[1])

    for ax in axs:
        mark_windows(ax)

    fig.suptitle(r"Slope statistics: mean and $\sigma$", fontsize=20)
    savefig(fig, "slope_stats_mean_sigma.png")
    plt.show()

In [ ]:
if HAS_SLOPES:
    fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
    ax.plot(tau, sx_skew, "-o", color="#1b9e77", label=r"$\gamma_1(s_x)$")
    ax.plot(tau, sz_skew, "-s", color="#e7298a", label=r"$\gamma_1(s_z)$")
    ax.axhline(0.0, color="0.4", lw=0.9, ls="--", alpha=0.7)
    ax.set_xlabel(r"$\tau = (t_{\mathrm{rel}} - t_{\mathrm{rel},0})/T_p$")
    ax.set_ylabel(r"skewness")
    ax.legend(loc="best", frameon=False, ncol=2)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
    style_axes(ax)
    mark_windows(ax)
    fig.suptitle(r"Skewness of surface slopes $\partial\eta/\partial x$ and $\partial\eta/\partial z$",
                 fontsize=18)
    savefig(fig, "slope_stats_skewness.png")
    plt.show()

In [ ]:
if HAS_SLOPES:
    fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
    ax.plot(tau, sx_kurt, "-o", color="#6a3d9a", label=r"$\kappa(s_x)$")
    ax.plot(tau, sz_kurt, "-s", color="#ff7f00", label=r"$\kappa(s_z)$")
    ax.axhline(3.0, color="0.4", lw=0.9, ls=":", alpha=0.8, label="Gaussian = 3")
    ax.set_xlabel(r"$\tau = (t_{\mathrm{rel}} - t_{\mathrm{rel},0})/T_p$")
    ax.set_ylabel(r"kurtosis")
    ax.legend(loc="best", frameon=False, ncol=3)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
    style_axes(ax)
    mark_windows(ax)
    fig.suptitle(r"Kurtosis of surface slopes $\partial\eta/\partial x$ and $\partial\eta/\partial z$",
                 fontsize=18)
    savefig(fig, "slope_stats_kurtosis.png")
    plt.show()

### Cross-check: elevation skewness against slope kurtosis

For a weakly nonlinear wave field the vertical asymmetry of the surface (sharp crests,
flat troughs) shows up as positive $\gamma_1(\eta)$, and the same sharpening makes the
slope distribution heavy-tailed, $\kappa(s_x) > 3$. Plotting both against $\tau$ on a
twin axis is a quick consistency test: the two should move together, and a case where
one drifts and the other does not points at a problem in the slope estimator rather
than at new physics.

In [ ]:
if HAS_SLOPES:
    fig, axa = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
    axb = axa.twinx()

    l1, = axa.plot(tau, xi_skew, "-o",  color="#1b9e77", lw=2.0, label=r"$\gamma_1(\eta)$")
    l2, = axa.plot(tau, sx_skew, "--^", color="#33a02c", lw=1.8, label=r"$\gamma_1(s_x)$")
    axa.axhline(0.0, color="0.5", lw=0.8, ls="--", alpha=0.6)
    axa.set_xlabel(r"$\tau = (t_{\mathrm{rel}} - t_{\mathrm{rel},0})/T_p$")
    axa.set_ylabel(r"skewness  $\gamma_1$", color="#1b9e77")
    axa.tick_params(axis="y", labelcolor="#1b9e77")

    l3, = axb.plot(tau, xi_kurt, "-s",  color="#6a3d9a", lw=2.0, label=r"$\kappa(\eta)$")
    l4, = axb.plot(tau, sx_kurt, "--v", color="#9e3d9a", lw=1.8, label=r"$\kappa(s_x)$")
    axb.axhline(3.0, color="0.55", lw=0.8, ls=":", alpha=0.7)
    axb.set_ylabel(r"kurtosis  $\kappa$", color="#6a3d9a")
    axb.tick_params(axis="y", labelcolor="#6a3d9a")

    axa.xaxis.set_major_locator(MaxNLocator(nbins=8))
    style_axes(axa)
    axa.legend(handles=[l1, l2, l3, l4], loc="best", frameon=False, ncol=4)
    mark_windows(axa)

    fig.suptitle(r"Cross-check: $\gamma_1(\eta)$ vs $\kappa(\partial\eta/\partial x)$", fontsize=18)
    savefig(fig, "slope_crosscheck_skew_kurt.png")
    plt.show()

## 4. Elevation PDFs

One file per window, columns
`0:eta_center 1:xi_center 2:pdf_xi 3:pdf_gauss 4:count 5:pdf_eta`.
The reference Gaussian in column 3 is the unit normal in $\xi$, so any departure is a
direct read of the non-Gaussianity of the surface.

In [ ]:
pdf_files = sorted(glob.glob(os.path.join(STATS_DIR, "eta_pdf_window_*.out")))
print(f"{len(pdf_files)} PDF window file(s)")

pdfs = []   # (window label, eta, pdf_eta, xi, pdf_xi, pdf_gauss)
for path in pdf_files:
    d = load_table(path)
    if d.size == 0 or d.shape[1] < 6:
        print(f"skipping empty/incomplete file: {os.path.basename(path)}")
        continue
    label = os.path.basename(path).replace(".out", "").replace("eta_pdf_window_", "Window ")
    pdfs.append((label, d[:, 0], d[:, 5], d[:, 1], d[:, 2], d[:, 3]))

print(f"{len(pdfs)} usable")

In [ ]:
# Which windows to draw individually: "latest", "all", or a list of indices into `pdfs`.
WINDOWS_TO_PLOT = "latest"

if not pdfs:
    print("No usable PDF files - skipping.")
else:
    if WINDOWS_TO_PLOT == "latest":
        sel = [len(pdfs) - 1]
    elif WINDOWS_TO_PLOT == "all":
        sel = range(len(pdfs))
    else:
        sel = WINDOWS_TO_PLOT

    for k in sel:
        label, eta, pdf_eta, xi, pdf_xi, pdf_gauss = pdfs[k]
        base = f"eta_pdf_window_{label.split()[-1]}"

        fig, ax = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
        ax.plot(xi, pdf_xi, color="#0b6e4f", lw=2.2, label=r"$p(\xi)$")
        ax.plot(xi, pdf_gauss, "--", color="black", lw=1.6, label=r"Gaussian")
        ax.fill_between(xi, pdf_xi, 0.0, color="#0b6e4f", alpha=0.10)
        ax.set_xlabel(r"$\xi$")
        ax.set_ylabel(r"PDF")
        ax.set_title(label)
        ax.legend(loc="best", frameon=False)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
        style_axes(ax)
        savefig(fig, f"{base}_xi.png")
        plt.show()

        fig, ax = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
        ax.plot(eta, pdf_eta, color="#1f4e79", lw=2.2, label=r"$p(\eta)$")
        ax.fill_between(eta, pdf_eta, 0.0, color="#1f4e79", alpha=0.10)
        ax.set_xlabel(r"$\eta$")
        ax.set_ylabel(r"PDF$(\eta)$")
        ax.set_title(label)
        ax.legend(loc="best", frameon=False)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
        style_axes(ax)
        savefig(fig, f"{base}_eta.png")
        plt.show()

### All windows at once

Overlaying every window on a log axis is the view the per-window PNGs cannot give:
window-to-window scatter in the tails shows how far the statistics are from converged,
and a systematic drift from early to late windows (colour) separates a transient from
noise.

In [ ]:
if pdfs:
    fig, axs = plt.subplots(1, 2, figsize=(13, 5.2), constrained_layout=True)
    colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(pdfs)))

    for (label, eta, pdf_eta, xi, pdf_xi, pdf_gauss), col in zip(pdfs, colors):
        m = pdf_xi > 0
        axs[0].semilogy(xi[m], pdf_xi[m], color=col, lw=1.4, alpha=0.85)
        m = pdf_eta > 0
        axs[1].semilogy(eta[m], pdf_eta[m], color=col, lw=1.4, alpha=0.85)

    # Reference Gaussian from the last window.
    xi_ref, g_ref = pdfs[-1][3], pdfs[-1][5]
    m = g_ref > 0
    axs[0].semilogy(xi_ref[m], g_ref[m], "--", color="k", lw=1.6, label="Gaussian")

    axs[0].set_xlabel(r"$\xi = (\eta - \langle\eta\rangle)/\sigma_\eta$")
    axs[0].set_ylabel(r"$p(\xi)$")
    axs[0].legend(loc="best", frameon=False)
    axs[1].set_xlabel(r"$\eta$")
    axs[1].set_ylabel(r"$p(\eta)$")
    for ax in axs:
        ax.xaxis.set_major_locator(MaxNLocator(nbins=7))
        style_axes(ax)

    sm = plt.cm.ScalarMappable(cmap="viridis",
                               norm=plt.Normalize(vmin=0, vmax=max(len(pdfs) - 1, 1)))
    cb = fig.colorbar(sm, ax=axs, pad=0.02)
    cb.set_label("window (early $\\rightarrow$ late)")

    fig.suptitle("Elevation PDFs, all windows", fontsize=18)
    savefig(fig, "eta_pdf_all_windows.png")
    plt.show()

## 5. Regenerating the PNG set

The figures above are written to `statistics/plots/` only when `SAVE_FIGS = True` in
section 1. To refresh the whole set headlessly instead — which is what the run script
does after every job — call the script directly:

```bash
python generate_eta_plots.py /path/to/case/statistics
```